# v11 Colab pilot — HaNoRec CF-hardness reranker

**Mục tiêu:** kiểm tra model/code trên GPU thật và đo pilot validation cho pipeline frozen LLM2Rec embeddings → frozen SASRec top-20 → Qwen2.5-VL reranker.

Pilot chạy **một** real-image SFT parent và hai DPO arm (`w=1.0`, semantic-only; `w=0.5`, semantic+CF). Chỉ dùng validation metrics; không chấm điểm test bằng Qwen. Một seed / một epoch là exploratory, không phải confirmation.

## Yêu cầu trước khi chạy

1. Colab Runtime → Change runtime type → GPU. Code v11 giữ protocol BF16; cần GPU báo `torch.cuda.is_bf16_supported() == True` (ví dụ L4/A100). T4 sẽ dừng rõ ràng, không âm thầm đổi sang FP16.
2. Không cần Google Drive. Đưa đủ 9 artifact đã pin vào `/content/v11_inputs/`: `data.txt`, `train_data.txt`, `val_data.txt`, `test_data.txt`, `item_titles.json`, `Video_Games_image_manifest.jsonl`, `games_sasrec_step500.log`, checkpoint SASRec, và LLM2Rec item embeddings. Cell input cho phép upload một ZIP hoặc các file rời; notebook xác minh SHA-256.
3. Chạy các cell theo thứ tự. Input, ảnh cache, checkpoint và kết quả được lưu ở `/content`; chúng mất khi Colab runtime reset. Tải archive kết quả về máy ở cell cuối.

## Giới hạn diễn giải

- `max_pixels=4096`, một epoch và một seed: chỉ pilot; không kết luận hiệu quả/không hiệu quả cuối cùng.
- Scoring Qwen chỉ chạy trên validation users mà target nằm trong SASRec top-20; các users còn lại được giữ là zero khi tính metric toàn cohort. Báo riêng số user informative và candidate recall.
- `prep.py` nguyên bản yêu cầu đọc test split để dựng object đầu vào; notebook xóa evaluation rows trước khi gọi model và không báo cáo test metrics. Không dùng test để chọn cấu hình.
- Không đổi công thức CF hardness, data split, model revision, hay precision.


In [ ]:
# Giữ nguyên PyTorch/CUDA Colab; KHÔNG nâng cấp torch riêng.
%pip -q install "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0" "pillow==11.0.0" requests


## 1. Cài mã nguồn v11 đã pin

Notebook chứa bản byte-identical của các module corrected tại thời điểm tạo file. Cell này ghi chúng vào `/content/v11_source` và xác minh hash trước khi import.


In [ ]:
import base64, hashlib, json, sys
from pathlib import Path

SOURCE_DIR = Path("/content/v11_source")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_SOURCES = {"prep.py":"...","train.py":"...","audit.py":"IiIiRmFpbC1jbG9zZWQgYXVkaXRzIGZvciB0aGUgY29ycmVjdGVkIEhhTm9SZWMgR2FtZXMgc3R1ZHkuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgcmFuZG9tCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKZGVmIF9sb2FkKHBhdGg6IFBhdGgpIC0+IEFueToKICAgIHJldHVybiBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQoKCmRlZiBwcm90b2NvbF9lcnJvcnMocHJvdG9jb2w6IGRpY3Rbc3RyLCBBbnldKSAtPiBsaXN0W3N0cl06CiAgICBlcnJvcnM6IGxpc3Rbc3RyXSA9IFtdCiAgICB0YXNrID0gcHJvdG9jb2wuZ2V0KCJ0YXNrIiwge30pCiAgICByZWNpcGUgPSBwcm90b2NvbC5nZXQoInJlY2lwZSIsIHt9KQogICAgc291cmNlID0gcHJvdG9jb2wuZ2V0KCJzb3VyY2VfYXV0aG9yaXR5Iiwge30pCiAgICBpZiBub3QgdGFzay5nZXQoInByaW1hcnlfY29udHJhc3RzIik6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgicHJpbWFyeSBjb250cmFzdHMgYXJlIG1pc3NpbmciKQogICAgaWYgZmxvYXQodGFzay5nZXQoIm1lYW5pbmdmdWxfbmRjZ19kZWx0YSIsIDApKSA8PSAwOgogICAgICAgIGVycm9ycy5hcHBlbmQoIm1lYW5pbmdmdWwgTkRDRyB0aHJlc2hvbGQgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBpZiBzb3VyY2UuZ2V0KCJoYW5vcmVjIiwge30pLmdldCgiY29tbWl0IikgIT0gIjU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiOgogICAgICAgIGVycm9ycy5hcHBlbmQoIkhhTm9SZWMgc291cmNlIGNvbW1pdCBpcyBub3QgcGlubmVkIikKICAgIGlmIHNvdXJjZS5nZXQoImxsbTJyZWMiLCB7fSkuZ2V0KCJjb21taXQiKSAhPSAiNzNiNDgxZjcxMGY2NzE2NmFiOTU4ZjQ5ODVkMjdiMjdmYjQxMDg3MSI6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgiTExNMlJlYyBzb3VyY2UgY29tbWl0IGlzIG5vdCBwaW5uZWQiKQogICAgZm9yIHN0YWdlIGluICgic2Z0IiwgImRwbyIpOgogICAgICAgIGNvbmZpZyA9IHJlY2lwZS5nZXQoc3RhZ2UsIHt9KQogICAgICAgIGZvciBmaWVsZCBpbiAoImJhdGNoX3NpemUiLCAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIiwgImVwb2NocyIsICJtYXhfZ3JhZF9ub3JtIik6CiAgICAgICAgICAgIGlmIGZpZWxkIG5vdCBpbiBjb25maWc6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie3N0YWdlfS57ZmllbGR9IGlzIG1pc3NpbmciKQogICAgICAgIGlmIHN0YWdlID09ICJzZnQiIGFuZCBjb25maWcuZ2V0KCJwcmVjaXNpb24iKSAhPSAiYmYxNiI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoIlNGVCBwcmVjaXNpb24gbXVzdCByZW1haW4gYmYxNiIpCiAgICBpZiByZWNpcGUuZ2V0KCJkcG8iLCB7fSkuZ2V0KCJyZWZlcmVuY2UiKSAhPSAiZnJvemVuIHBvc3QtU0ZUIGFkYXB0ZXIsIG5vIERQTyBhZGFwdGVyIjoKICAgICAgICBlcnJvcnMuYXBwZW5kKCJEUE8gcmVmZXJlbmNlIGNvbnRyYWN0IGlzIG5vdCBmcm96ZW4iKQogICAgaWYgcHJvdG9jb2wuZ2V0KCJzZWVkcyIpICE9IFsyMDI0LCAyMDI1LCAyMDI2XToKICAgICAgICBlcnJvcnMuYXBwZW5kKCJ0cmFpbmluZyBzZWVkcyBtdXN0IGJlIGV4YWN0bHkgMjAyNCwgMjAyNSwgMjAyNiIpCiAgICBpZiBzb3J0ZWQocHJvdG9jb2wuZ2V0KCJ3ZWlnaHRzIiwgW10pKSAhPSBbMC4wLCAwLjUsIDEuMF06CiAgICAgICAgZXJyb3JzLmFwcGVuZCgid2VpZ2h0cyBtdXN0IGNvbnRhaW4gMCwgMC41LCBhbmQgMSIpCiAgICBpZiAiaW52YXJpYW50cyIgbm90IGluIHByb3RvY29sIG9yIGxlbihwcm90b2NvbFsiaW52YXJpYW50cyJdKSA8IDg6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgicHJvdG9jb2wgaW52YXJpYW50cyBhcmUgaW5jb21wbGV0ZSIpCiAgICBzbW9rZSA9IHByb3RvY29sLmdldCgic21va2UiLCB7fSkKICAgIGlmIG5vdCBzbW9rZS5nZXQoIm11c3Rfbm90X2JlX3VzZWRfYXNfc2lnbmFsIiwgRmFsc2UpOgogICAgICAgIGVycm9ycy5hcHBlbmQoInNtb2tlIG11c3QgYmUgZXhwbGljaXRseSBub24tc2lnbmFsIikKICAgIHJldHVybiBlcnJvcnMKCgpkZWYgYXVkaXRfcHJvdG9jb2wocGF0aDogUGF0aCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBwcm90b2NvbCA9IF9sb2FkKHBhdGgpCiAgICBlcnJvcnMgPSBwcm90b2NvbF9lcnJvcnMocHJvdG9jb2wpCiAgICByZXN1bHQgPSB7InN0YXR1cyI6ICJQQVNTIiBpZiBub3QgZXJyb3JzIGVsc2UgIkZBSUwiLCAiZXJyb3JzIjogZXJyb3JzLCAicHJvdG9jb2wiOiBzdHIocGF0aCl9CiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBfZmluZChyb290OiBQYXRoLCBuYW1lOiBzdHIpIC0+IGxpc3RbUGF0aF06CiAgICByZXR1cm4gc29ydGVkKHBhdGggZm9yIHBhdGggaW4gcm9vdC5yZ2xvYihuYW1lKSBpZiBwYXRoLmlzX2ZpbGUoKSkKCgpkZWYgYXVkaXRfYXJ0aWZhY3RzKHJvb3Q6IFBhdGgsIHByb3RvY29sOiBkaWN0W3N0ciwgQW55XSkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXF1aXJlZCA9IFsic2Z0X2J1bmRsZS5qc29uIiwgInNmdF9sb3JhX3N0YXRlLnB0Il0KICAgIG1pc3NpbmcgPSBbbmFtZSBmb3IgbmFtZSBpbiByZXF1aXJlZCBpZiBub3QgX2ZpbmQocm9vdCwgbmFtZSldCiAgICBhcm1fcmVzdWx0cyA9IF9maW5kKHJvb3QsICJhcm1fcmVzdWx0Lmpzb24iKQogICAgZXJyb3JzID0gW2YibWlzc2luZyB7bmFtZX0iIGZvciBuYW1lIGluIG1pc3NpbmddCiAgICBleHBlY3RlZCA9IHsKICAgICAgICAoZmxvYXQod2VpZ2h0KSwgY29uZGl0aW9uKQogICAgICAgIGZvciB3ZWlnaHQgaW4gcHJvdG9jb2xbIndlaWdodHMiXQogICAgICAgIGZvciBjb25kaXRpb24gaW4gcHJvdG9jb2xbImNvbnRyb2xzIl1bImltYWdlX2NvbmRpdGlvbnMiXQogICAgfQogICAgc21va2VfbWFuaWZlc3QgPSByb290IC8gInNtb2tlX21hbmlmZXN0Lmpzb24iCiAgICBpZiBzbW9rZV9tYW5pZmVzdC5leGlzdHMoKToKICAgICAgICBzbW9rZSA9IF9sb2FkKHNtb2tlX21hbmlmZXN0KQogICAgICAgIGlmIHNtb2tlLmdldCgibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiKSBpcyBub3QgVHJ1ZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZCgic21va2UgbWFuaWZlc3QgaXMgbm90IGV4cGxpY2l0bHkgbm9uLXNpZ25hbCIpCiAgICAgICAgaWYgc21va2UuZ2V0KCJwdXJwb3NlIikgIT0gImNvcnJlY3RuZXNzX29ubHlfbm9uX3NpZ25hbCI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoInNtb2tlIHB1cnBvc2UgaXMgbm90IGNvcnJlY3RuZXNzLW9ubHkiKQogICAgICAgIGV4cGVjdGVkID0geygxLjAsICJyZWFsIil9CiAgICAgICAgZm9yIGxhYmVsIGluICgic2Z0X2dwdV9tZW1vcnkiLCAiZHBvX2dwdV9tZW1vcnkiKToKICAgICAgICAgICAgbWVtb3J5ID0gc21va2UuZ2V0KGxhYmVsLCB7fSkKICAgICAgICAgICAgdG90YWwgPSBpbnQobWVtb3J5LmdldCgidG90YWxfYnl0ZXMiLCAwKSkKICAgICAgICAgICAgYWxsb2NhdGVkID0gaW50KG1lbW9yeS5nZXQoInBlYWtfYWxsb2NhdGVkX2J5dGVzIiwgMCkpCiAgICAgICAgICAgIHJlc2VydmVkID0gaW50KG1lbW9yeS5nZXQoInBlYWtfcmVzZXJ2ZWRfYnl0ZXMiLCAwKSkKICAgICAgICAgICAgaWYgbm90ICgwIDwgYWxsb2NhdGVkIDw9IHJlc2VydmVkIDw9IHRvdGFsKToKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJpbnZhbGlkIHtsYWJlbH0gdGVsZW1ldHJ5IikKICAgICAgICBmb3IgbGFiZWwsIHByZWZpeCBpbiAoKCJzZnRfc3RhZ2VfdGltaW5ncyIsICJzZnRfdXBkYXRlIiksICgiZHBvX3N0YWdlX3RpbWluZ3MiLCAiZHBvX3VwZGF0ZV8iKSk6CiAgICAgICAgICAgIHRpbWluZ3MgPSBzbW9rZS5nZXQobGFiZWwsIFtdKQogICAgICAgICAgICB1cGRhdGVzID0gWwogICAgICAgICAgICAgICAgcm93IGZvciByb3cgaW4gdGltaW5ncwogICAgICAgICAgICAgICAgaWYgc3RyKHJvdy5nZXQoInN0YWdlIiwgIiIpKS5zdGFydHN3aXRoKHByZWZpeCkKICAgICAgICAgICAgXQogICAgICAgICAgICBpZiBub3QgdXBkYXRlcyBvciBhbnkoZmxvYXQocm93LmdldCgic2Vjb25kcyIsIDApKSA8PSAwIGZvciByb3cgaW4gdXBkYXRlcyk6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiaW52YWxpZCB7bGFiZWx9IG9wdGltaXplciB0aW1pbmciKQogICAgb2JzZXJ2ZWQ6IHNldFt0dXBsZVtmbG9hdCwgc3RyXV0gPSBzZXQoKQogICAgZm9yIHBhdGggaW4gYXJtX3Jlc3VsdHM6CiAgICAgICAgcmVzdWx0ID0gX2xvYWQocGF0aCkKICAgICAgICBrZXkgPSAoZmxvYXQocmVzdWx0LmdldCgid2VpZ2h0IiwgZmxvYXQoIm5hbiIpKSksIHN0cihyZXN1bHQuZ2V0KCJpbWFnZV9jb25kaXRpb24iLCAiIikpKQogICAgICAgIGlmIGtleSBpbiBvYnNlcnZlZDoKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmImR1cGxpY2F0ZSBhcm06IHtrZXl9IikKICAgICAgICBvYnNlcnZlZC5hZGQoa2V5KQogICAgICAgIGlmIHJlc3VsdC5nZXQoInN0YXR1cyIpICE9ICJDT01QTEVURSI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJpbmNvbXBsZXRlIGFybToge3BhdGh9IikKICAgICAgICBwcmVkaWN0aW9ucyA9IHJlc3VsdC5nZXQoInByZWRpY3Rpb25zIiwgW10pCiAgICAgICAgaWYgbm90IHByZWRpY3Rpb25zOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiYXJtIGhhcyBubyBwcmVkaWN0aW9uczoge3BhdGh9IikKICAgICAgICBmb3Igcm93IGluIHByZWRpY3Rpb25zOgogICAgICAgICAgICBmb3IgbWV0cmljIGluICgibmRjZ0AxMCIsICJyZWNhbGxAMTAiLCAiY2FuZGlkYXRlX3JlY2FsbEAyMCIpOgogICAgICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoZmxvYXQocm93LmdldChtZXRyaWMsIGZsb2F0KCJuYW4iKSkpKToKICAgICAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYibm9uLWZpbml0ZSB7bWV0cmljfToge3BhdGh9IikKICAgICAgICBjaGVja3BvaW50ID0gcmVzdWx0LmdldCgiY2hlY2twb2ludCIpCiAgICAgICAgY2hlY2twb2ludF9wYXRoID0gKAogICAgICAgICAgICBwYXRoLnBhcmVudCAvIGNoZWNrcG9pbnQKICAgICAgICAgICAgaWYgY2hlY2twb2ludCBhbmQgbm90IFBhdGgoY2hlY2twb2ludCkuaXNfYWJzb2x1dGUoKQogICAgICAgICAgICBlbHNlIFBhdGgoY2hlY2twb2ludCkgaWYgY2hlY2twb2ludCBlbHNlIE5vbmUKICAgICAgICApCiAgICAgICAgaWYgY2hlY2twb2ludF9wYXRoIGlzIE5vbmUgb3Igbm90IGNoZWNrcG9pbnRfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgY2hlY2twb2ludDoge3BhdGh9IikKICAgICAgICBpZiBub3QgcmVzdWx0LmdldCgicGFyZW50X3NmdF9zaGEyNTYiKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgcGFyZW50IGhhc2g6IHtwYXRofSIpCiAgICBpZiBvYnNlcnZlZCAhPSBleHBlY3RlZDoKICAgICAgICBlcnJvcnMuYXBwZW5kKGYiYXJtIG1hdHJpeCBtaXNtYXRjaDogZXhwZWN0ZWQge3NvcnRlZChleHBlY3RlZCl9LCBvYnNlcnZlZCB7c29ydGVkKG9ic2VydmVkKX0iKQogICAgb3V0cHV0ID0gewogICAgICAgICJzdGF0dXMiOiAiUEFTUyIgaWYgbm90IGVycm9ycyBlbHNlICJGQUlMIiwKICAgICAgICAiZXJyb3JzIjogZXJyb3JzLAogICAgICAgICJzZnRfYnVuZGxlX2NvdW50IjogbGVuKF9maW5kKHJvb3QsICJzZnRfYnVuZGxlLmpzb24iKSksCiAgICAgICAgImFybV9yZXN1bHRfY291bnQiOiBsZW4oYXJtX3Jlc3VsdHMpLAogICAgICAgICJleHBlY3RlZF9hcm1fY291bnQiOiBsZW4oZXhwZWN0ZWQpLAogICAgfQogICAgcHJpbnQoanNvbi5kdW1wcyhvdXRwdXQsIGluZGVudD0yKSkKICAgIHJldHVybiBvdXRwdXQKCgpkZWYgX3BhaXJlZF9tZWFuKHJvd3NfYTogbGlzdFtkaWN0W3N0ciwgQW55XV0sIHJvd3NfYjogbGlzdFtkaWN0W3N0ciwgQW55XV0sIG1ldHJpYzogc3RyKSAtPiBmbG9hdDoKICAgIGJ5X2tleSA9IHsocm93WyJ1c2VyX2lkIl0sIHJvd1sidGFyZ2V0Il0pOiByb3cgZm9yIHJvdyBpbiByb3dzX2J9CiAgICBkZWx0YXMgPSBbXQogICAgZm9yIHJvdyBpbiByb3dzX2E6CiAgICAgICAgb3RoZXIgPSBieV9rZXkuZ2V0KChyb3dbInVzZXJfaWQiXSwgcm93WyJ0YXJnZXQiXSkpCiAgICAgICAgaWYgb3RoZXIgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGFpcmVkIHByZWRpY3Rpb24ga2V5cyBkaWZmZXIiKQogICAgICAgIGRlbHRhcy5hcHBlbmQoZmxvYXQocm93W21ldHJpY10pIC0gZmxvYXQob3RoZXJbbWV0cmljXSkpCiAgICBpZiBub3QgZGVsdGFzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vIHBhaXJlZCBwcmVkaWN0aW9ucyIpCiAgICByZXR1cm4gc3VtKGRlbHRhcykgLyBsZW4oZGVsdGFzKQoKZGVmIF9wYWlyZWRfZGVsdGFzKHJvd3NfYTogbGlzdFtkaWN0W3N0ciwgQW55XV0sIHJvd3NfYjogbGlzdFtkaWN0W3N0ciwgQW55XV0sIG1ldHJpYzogc3RyKSAtPiBsaXN0W2Zsb2F0XToKICAgIGJ5X2tleSA9IHsocm93WyJ1c2VyX2lkIl0sIHJvd1sidGFyZ2V0Il0pOiByb3cgZm9yIHJvdyBpbiByb3dzX2J9CiAgICBkZWx0YXMgPSBbXQogICAgZm9yIHJvdyBpbiByb3dzX2E6CiAgICAgICAgb3RoZXIgPSBieV9rZXkuZ2V0KChyb3dbInVzZXJfaWQiXSwgcm93WyJ0YXJnZXQiXSkpCiAgICAgICAgaWYgb3RoZXIgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGFpcmVkIHByZWRpY3Rpb24ga2V5cyBkaWZmZXIiKQogICAgICAgIGRlbHRhcy5hcHBlbmQoZmxvYXQocm93W21ldHJpY10pIC0gZmxvYXQob3RoZXJbbWV0cmljXSkpCiAgICBpZiBub3QgZGVsdGFzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vIHBhaXJlZCBwcmVkaWN0aW9ucyIpCiAgICByZXR1cm4gZGVsdGFzCgoKZGVmIF9ib290c3RyYXAoZGVsdGFzOiBsaXN0W2Zsb2F0XSwgcmVzYW1wbGVzOiBpbnQsIHNlZWQ6IGludCkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG1lYW5zID0gWwogICAgICAgIHN1bShkZWx0YXNbcm5nLnJhbmRyYW5nZShsZW4oZGVsdGFzKSldIGZvciBfIGluIGRlbHRhcykgLyBsZW4oZGVsdGFzKQogICAgICAgIGZvciBfIGluIHJhbmdlKHJlc2FtcGxlcykKICAgIF0KICAgIG1lYW5zLnNvcnQoKQogICAgbG93ID0gbWVhbnNbbWF4KDAsIGludCgwLjAyNSAqIHJlc2FtcGxlcykgLSAxKV0KICAgIGhpZ2ggPSBtZWFuc1ttaW4ocmVzYW1wbGVzIC0gMSwgaW50KDAuOTc1ICogcmVzYW1wbGVzKSldCiAgICBwID0gbWluKDEuMCwgMi4wICogbWluKAogICAgICAgIHN1bSh2YWx1ZSA8PSAwLjAgZm9yIHZhbHVlIGluIG1lYW5zKSAvIHJlc2FtcGxlcywKICAgICAgICBzdW0odmFsdWUgPj0gMC4wIGZvciB2YWx1ZSBpbiBtZWFucykgLyByZXNhbXBsZXMsCiAgICApKQogICAgcmV0dXJuIHsibWVhbiI6IHN1bShkZWx0YXMpIC8gbGVuKGRlbHRhcyksICJjaV9sb3ciOiBsb3csICJjaV9oaWdoIjogaGlnaCwgInBfdmFsdWUiOiBwfQoKCmRlZiBhbmFseXplKHJvb3Q6IFBhdGgsIGJvb3RzdHJhcF9yZXNhbXBsZXM6IGludCwgc2VlZDogaW50KSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGlmIGJvb3RzdHJhcF9yZXNhbXBsZXMgPCAxMDAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJvb3RzdHJhcF9yZXNhbXBsZXMgbXVzdCBiZSBhdCBsZWFzdCAxMDAwIikKICAgIGFybXM6IGRpY3RbdHVwbGVbZmxvYXQsIHN0cl0sIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICBmb3IgcGF0aCBpbiBfZmluZChyb290LCAiYXJtX3Jlc3VsdC5qc29uIik6CiAgICAgICAgcmVzdWx0ID0gX2xvYWQocGF0aCkKICAgICAgICBrZXkgPSAoZmxvYXQocmVzdWx0WyJ3ZWlnaHQiXSksIHN0cihyZXN1bHRbImltYWdlX2NvbmRpdGlvbiJdKSkKICAgICAgICBpZiBrZXkgaW4gYXJtczoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiZHVwbGljYXRlIGFybSB7a2V5fSIpCiAgICAgICAgYXJtc1trZXldID0gcmVzdWx0CiAgICByZXF1aXJlZCA9IFsKICAgICAgICAoMC4wLCAicmVhbCIpLCAoMC41LCAicmVhbCIpLCAoMS4wLCAicmVhbCIpLAogICAgICAgICgwLjAsICJzaHVmZmxlIiksICgwLjUsICJzaHVmZmxlIiksICgxLjAsICJzaHVmZmxlIiksCiAgICBdCiAgICBtaXNzaW5nID0gW2tleSBmb3Iga2V5IGluIHJlcXVpcmVkIGlmIGtleSBub3QgaW4gYXJtc10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYibWlzc2luZyBmaW5hbCBhcm1zOiB7bWlzc2luZ30iKQogICAgY29udHJhc3RzID0gW10KICAgIGZvciBpbmRleCwgd2VpZ2h0IGluIGVudW1lcmF0ZSgoMC4wLCAwLjUpKToKICAgICAgICByZWFsID0gYXJtc1sod2VpZ2h0LCAicmVhbCIpXVsicHJlZGljdGlvbnMiXQogICAgICAgIGJhc2VsaW5lID0gYXJtc1soMS4wLCAicmVhbCIpXVsicHJlZGljdGlvbnMiXQogICAgICAgIG5kY2cgPSBfYm9vdHN0cmFwKF9wYWlyZWRfZGVsdGFzKHJlYWwsIGJhc2VsaW5lLCAibmRjZ0AxMCIpLCBib290c3RyYXBfcmVzYW1wbGVzLCBzZWVkICsgaW5kZXgpCiAgICAgICAgcmVjYWxsID0gX2Jvb3RzdHJhcChfcGFpcmVkX2RlbHRhcyhyZWFsLCBiYXNlbGluZSwgInJlY2FsbEAxMCIpLCBib290c3RyYXBfcmVzYW1wbGVzLCBzZWVkICsgMTAwICsgaW5kZXgpCiAgICAgICAgY29udHJhc3RzLmFwcGVuZCh7IndlaWdodCI6IHdlaWdodCwgImNvbnRyYXN0IjogInctdnMtdzEiLCAibmRjZ0AxMCI6IG5kY2csICJyZWNhbGxAMTAiOiByZWNhbGx9KQogICAgcF92YWx1ZXMgPSBzb3J0ZWQoKGl0ZW1bIm5kY2dAMTAiXVsicF92YWx1ZSJdLCBpbmRleCkgZm9yIGluZGV4LCBpdGVtIGluIGVudW1lcmF0ZShjb250cmFzdHMpKQogICAgaG9sbSA9IHt9CiAgICB0b3RhbCA9IGxlbihwX3ZhbHVlcykKICAgIGZvciByYW5rLCAocF92YWx1ZSwgaW5kZXgpIGluIGVudW1lcmF0ZShwX3ZhbHVlcyk6CiAgICAgICAgaG9sbVtzdHIoY29udHJhc3RzW2luZGV4XVsid2VpZ2h0Il0pXSA9IG1pbigxLjAsICh0b3RhbCAtIHJhbmspICogcF92YWx1ZSkKICAgIG91dHB1dCA9IHsKICAgICAgICAic3RhdHVzIjogIkNPTVBMRVRFIiwKICAgICAgICAiYm9vdHN0cmFwX3Jlc2FtcGxlcyI6IGJvb3RzdHJhcF9yZXNhbXBsZXMsCiAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICJob2xtX2FkanVzdGVkX25kY2dfcCI6IGhvbG0sCiAgICAgICAgImNvbnRyYXN0cyI6IGNvbnRyYXN0cywKICAgIH0KICAgIHByaW50KGpzb24uZHVtcHMob3V0cHV0LCBpbmRlbnQ9MikpCiAgICByZXR1cm4gb3V0cHV0CgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBzdWJwYXJzZXJzID0gcGFyc2VyLmFkZF9zdWJwYXJzZXJzKGRlc3Q9ImNvbW1hbmQiLCByZXF1aXJlZD1UcnVlKQogICAgcHJvdG9jb2xfcGFyc2VyID0gc3VicGFyc2Vycy5hZGRfcGFyc2VyKCJwcm90b2NvbCIpCiAgICBwcm90b2NvbF9wYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb3RvY29sIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJ0aWZhY3RzX3BhcnNlciA9IHN1YnBhcnNlcnMuYWRkX3BhcnNlcigiYXJ0aWZhY3RzIikKICAgIGFydGlmYWN0c19wYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb3RvY29sIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJ0aWZhY3RzX3BhcnNlci5hZGRfYXJndW1lbnQoIi0tcm9vdCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFuYWx5emVfcGFyc2VyID0gc3VicGFyc2Vycy5hZGRfcGFyc2VyKCJhbmFseXplIikKICAgIGFuYWx5emVfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm90b2NvbCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFuYWx5emVfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYW5hbHl6ZV9wYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvb3RzdHJhcC1yZXNhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwMCkKICAgIGFuYWx5emVfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAyNjA5MjApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgaWYgYXJncy5jb21tYW5kID09ICJwcm90b2NvbCI6CiAgICAgICAgcmVzdWx0ID0gYXVkaXRfcHJvdG9jb2woYXJncy5wcm90b2NvbCkKICAgIGVsaWYgYXJncy5jb21tYW5kID09ICJhcnRpZmFjdHMiOgogICAgICAgIHByb3RvY29sX3Jlc3VsdCA9IGF1ZGl0X3Byb3RvY29sKGFyZ3MucHJvdG9jb2wpCiAgICAgICAgaWYgcHJvdG9jb2xfcmVzdWx0WyJzdGF0dXMiXSAhPSAiUEFTUyI6CiAgICAgICAgICAgIHJldHVybiAxCiAgICAgICAgcmVzdWx0ID0gYXVkaXRfYXJ0aWZhY3RzKGFyZ3Mucm9vdCwgX2xvYWQoYXJncy5wcm90b2NvbCkpCiAgICBlbHNlOgogICAgICAgIHByb3RvY29sX3Jlc3VsdCA9IGF1ZGl0X3Byb3RvY29sKGFyZ3MucHJvdG9jb2wpCiAgICAgICAgaWYgcHJvdG9jb2xfcmVzdWx0WyJzdGF0dXMiXSAhPSAiUEFTUyI6CiAgICAgICAgICAgIHJldHVybiAxCiAgICAgICAgcmVzdWx0ID0gYW5hbHl6ZShhcmdzLnJvb3QsIGFyZ3MuYm9vdHN0cmFwX3Jlc2FtcGxlcywgYXJncy5zZWVkKQogICAgcmV0dXJuIDAgaWYgcmVzdWx0WyJzdGF0dXMiXSBpbiB7IlBBU1MiLCAiQ09NUExFVEUifSBlbHNlIDEKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==","experiment.json":"ewogICJzZWVkIjogMjAyNCwKICAidG9wX20iOiAyMCwKICAidHJhaW5fcGFpcnMiOiA1MzAsCiAgImV2YWxfdXNlcnMiOiAyNjUsCiAgInZhbGlkYXRpb25fdXNlcnMiOiAyNjUsCiAgImhpc3RvcnlfaXRlbXMiOiAzLAogICJiYXRjaF9zaXplIjogNCwKICAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIjogOCwKICAibGVhcm5pbmdfcmF0ZSI6IDAuMDAwMSwKICAic2Z0X2Vwb2NocyI6IDUsCiAgImRwb19lcG9jaHMiOiA1LAogICJkcm9wX2xhc3QiOiB0cnVlLAogICJwcmVjaXNpb24iOiAiYmYxNiIsCiAgIm1vZGVsX2lkIjogIlF3ZW4vUXdlbjIuNS1WTC0zQi1JbnN0cnVjdCIsCiAgIm1vZGVsX3JldmlzaW9uIjogIjY2Mjg1NTQ2ZDJiODIxY2Y0MjFkNGY1ZWIyNTc2MzU5ZDM3NzBjZDMiLAogICJ3ZWlnaHRzIjogWzEuMCwgMC4wLCAwLjVdLAogICJub2lzZV9zaWdtYSI6IDAuMDUsCiAgImJldGEwIjogMC4xLAogICJiZXRhX2Zsb29yIjogMS4wZS02LAogICJtYXhfcGl4ZWxzIjogMjYyMTQ0LAogICJidWRnZXRfc2Vjb25kcyI6IDY2MDAsCiAgInVwc3RyZWFtIjogewogICAgImxsbTJyZWMiOiB7CiAgICAgICJ1cmwiOiAiaHR0cHM6Ly9jb2RlbG9hZC5naXRodWIuY29tL0hhcHB5UG9pbnRlci9MTE0yUmVjL3ppcC83M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIiwKICAgICAgInNoYTI1NiI6ICI2YmNlZTdiMDllYzlhZDVmYTAxMDdhMDgzYzdiODA2MGFiYmVhZDIzZTEyNjRhNDk0ODA5MDZkZTYzOGMxY2M4IiwKICAgICAgImNvbW1pdCI6ICI3M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIgogICAgfSwKICAgICJoYW5vcmVjIjogewogICAgICAidXJsIjogImh0dHBzOi8vY29kZWxvYWQuZ2l0aHViLmNvbS93YW5neXUwNjI3L0hhTm9SZWMvemlwLzU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiLAogICAgICAic2hhMjU2IjogIjQ4YzUyZTU2MjVjZDRlMjVlODA2MDBmZWQ5YmVkOGFhZDg5MDY1NTZkMjY3Yjg3ZDUyODkwYjQ5YmYzM2Q2YTYiLAogICAgICAiY29tbWl0IjogIjU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiCiAgICB9CiAgfSwKICAiYXJ0aWZhY3RfcGlucyI6IHsKICAgICJkYXRhLnR4dCI6ICI0OGM3ZWFkOThhYmZhNzNjNTA2YmY3YWYxMzM3MDNmMmVlN2U5MjIxNjdkOTIyNzYwYmIwNzQzNzY2MzlhNzJjIiwKICAgICJ0cmFpbl9kYXRhLnR4dCI6ICIxMzAwZTVkZWVjMjlkNGJlZGU2ZTMyYmZhMWE1ZWFkZTU2M2Y1M2E2OTE2ZGE5NWQ2Nzc1NGVjNDBiOTQ3NWI4IiwKICAgICJ2YWxfZGF0YS50eHQiOiAiNDg0YTk3Y2ZiY2ZmMmE3OGU5MjYxMmIxNTg2M2NlMDhjODRjMjg1YzdiZmIzMjVlYzFmMzNkMzg4NGU2ZGUxNiIsCiAgICAidGVzdF9kYXRhLnR4dCI6ICI4MDA3NGZmZWEzNzkyOGQ5MmMzNTAzOGU5ZDhiYmYzZGUwY2ZjZjAxMzA4YWU0MTczZTNjNjgwYzFiYzUwZDFmIiwKICAgICJpdGVtX3RpdGxlcy5qc29uIjogIjE3ZjUwMTgwOWM0MTU5OTA1Y2ZhNmRmYTU2MjZkM2Q1MmQwYmUzMmJjOTc0N2VlODliMDAzNjYyZjZmNWU4NGMiLAogICAgIkxMTTJSZWMtYnVkZ2V0ZWQtfGthZ2dsZXx3b3JraW5nfExMTTJSZWN8cmVwZWF0ZWRfZXZhbHVhdGVfd2l0aF9zZXEtU2VwLTA5LTIwMjZfMTEtMTItMTMtYzkzZjEyLnB0aCI6ICJmYjBkMWQzOTE2ZjEwYjViNmYwZDlmNTk3ZTdiMmFjODQzN2Y0NDVjMTU2MjllYmYzMDk5MzMzYjcyODA2YTU5IiwKICAgICJRd2VuMi0wLjVCLUxMTTJSZWMtSUVNLWJ1ZGdldGVkX3N0ZXA1MDBfdGl0bGVfaXRlbV9lbWJzLm5weSI6ICI4OWY5ZDNkZDE3ZjllYzQ5NTM3ZjkxMGUzOWM5NzYxODc4Mjk5ZjdkMjRhYmRjZjgwNWFiMGRiOTE0MzNjMGYwIiwKICAgICJnYW1lc19zYXNyZWNfc3RlcDUwMC5sb2ciOiAiYTYwMTgxYTI2MTE0MDcyZGQ1NDQyZmQ5ZmZlNGIzMzg1MDUwMTBkOGIzMjFkOWUzMWNhMTY0NmIzNDFmZjE1NCIsCiAgICAiVmlkZW9fR2FtZXNfaW1hZ2VfbWFuaWZlc3QuanNvbmwiOiAiNWQ5YzU4NmQyZDA1YzZmZjIwOGM3YTBmOTQ1YTQ0MWE0ZmM2MGVmMGQxOGVjOGFhY2MyNjlmYmY2N2E0MmQwNCIKICB9LAogICJjaGVja3BvaW50X25hbWUiOiAiTExNMlJlYy1idWRnZXRlZC18a2FnZ2xlfHdvcmtpbmd8TExNMlJlY3xyZXBlYXRlZF9ldmFsdWF0ZV93aXRoX3NlcS1TZXAtMDktMjAyNl8xMS0xMi0xMy1jOTNmMTIucHRoIiwKICAiZW1iZWRkaW5nX25hbWUiOiAiUXdlbjItMC41Qi1MTE0yUmVjLUlFTS1idWRnZXRlZF9zdGVwNTAwX3RpdGxlX2l0ZW1fZW1icy5ucHkiLAogICJwdXJwb3NlIjogImNvcnJlY3RlZCBHYW1lcyBhZGFwdGF0aW9uOyBmdWxsIHNpZ25hbCByZXF1aXJlcyBQaGFzZSAzIHF1b3RhIGdhdGUiCn0K","run-protocol.json":"ewogICJwcm90b2NvbF92ZXJzaW9uIjogMSwKICAicGxhbiI6ICIyNjA5MjAtMTI0NS1oYW5vcmVjLWZhaXRoZnVsLXNpZ25hbC1yZWNoZWNrIiwKICAidGFzayI6IHsKICAgICJkb21haW4iOiAiR2FtZXMgYWRhcHRhdGlvbiBvZiBIYU5vUmVjIGhpdC0xIGJpbmFyeSBwcmVmZXJlbmNlIiwKICAgICJuYXRpdmVfc2NvcmVfY2xhaW0iOiBmYWxzZSwKICAgICJwcmltYXJ5X21ldHJpY3MiOiBbIk5EQ0dAMTAiLCAiUmVjYWxsQDEwIl0sCiAgICAicHJpbWFyeV9jb250cmFzdHMiOiBbInc9MC4wIG1pbnVzIHc9MS4wIiwgInc9MC41IG1pbnVzIHc9MS4wIl0sCiAgICAibWVhbmluZ2Z1bF9uZGNnX2RlbHRhIjogMC4wMDIsCiAgICAiY29uZmlybWF0aW9uX3NwbGl0IjogInRlc3QiLAogICAgImRldmVsb3BtZW50X3NwbGl0IjogInZhbGlkYXRpb24iLAogICAgImNhbmRpZGF0ZV9zb3VyY2UiOiAiZnJvemVuIFNBU1JlYyB0b3AtTSBjYW5kaWRhdGVzOyB0YXJnZXQgbmV2ZXIgaW5zZXJ0ZWQiLAogICAgInNjb3JpbmciOiAic2VxdWVuY2Utc3VtbWVkIFllcy9ObyByZXNwb25zZSBsb2ctcHJvYmFiaWxpdHkgbWFyZ2luIgogIH0sCiAgInNvdXJjZV9hdXRob3JpdHkiOiB7CiAgICAiaGFub3JlYyI6IHsKICAgICAgInJlcG9zaXRvcnkiOiAiaHR0cHM6Ly9naXRodWIuY29tL3dhbmd5dTA2MjcvSGFOb1JlYyIsCiAgICAgICJjb21taXQiOiAiNTg3ZmFjZTc0NTI0ZTQ1NTNiNWE3YWEyOTVmZTk2MjAwNDY4MjM4MiIsCiAgICAgICJhcmNoaXZlX3NoYTI1NiI6ICI0OGM1MmU1NjI1Y2Q0ZTI1ZTgwNjAwZmVkOWJlZDhhYWQ4OTA2NTU2ZDI2N2I4N2Q1Mjg5MGI0OWJmMzNkNmE2IiwKICAgICAgInBhdGhzIjogWyJjb25maWdzL2hhbm9yZWMvbWljcm9sZW5zX2hpdDEueWFtbCIsICJoYW5vcmVjL3RyYWluZXIvZHBvLnB5IiwgImhhbm9yZWMvaGFycy9tYXRoLnB5IiwgImhhbm9yZWMvaGFycy9oYXJkbmVzcy5weSIsICJoYW5vcmVjL25vZG8vaG9va3MucHkiXQogICAgfSwKICAgICJsbGFtYWZhY3RvcnkiOiB7CiAgICAgICJyZXF1aXJlbWVudCI6ICJsbGFtYWZhY3Rvcnk9PTAuOS4zLmRldjAiLAogICAgICAic291cmNlX3JldmlzaW9uIjogbnVsbCwKICAgICAgIm5vdGUiOiAiVGhlIHBpbm5lZCBIYU5vUmVjIHJlcXVpcmVtZW50IGV4cG9zZXMgYSBwYWNrYWdlIHZlcnNpb24gYnV0IG5vIHNvdXJjZSBjb21taXQuIFRoZSBjb3JyZWN0ZWQgR2FtZXMgcnVubmVyIHBvcnRzIGFuZCB0ZXN0cyB0aGUgb2JzZXJ2YWJsZSBIYU5vUmVjIHRyYWluZXIgY29udHJhY3Q7IGl0IGRvZXMgbm90IGNsYWltIGJ5dGUtaWRlbnRpY2FsIExMYU1BLUZhY3RvcnkgaW50ZXJuYWxzLiIKICAgIH0sCiAgICAibGxtMnJlYyI6IHsKICAgICAgInJlcG9zaXRvcnkiOiAiaHR0cHM6Ly9naXRodWIuY29tL0hhcHB5UG9pbnRlci9MTE0yUmVjIiwKICAgICAgImNvbW1pdCI6ICI3M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIiwKICAgICAgImFyY2hpdmVfc2hhMjU2IjogIjZiY2VlN2IwOWVjOWFkNWZhMDEwN2EwODNjN2I4MDYwYWJiZWFkMjNlMTI2NGE0OTQ4MDkwNmRlNjM4YzFjYzgiCiAgICB9CiAgfSwKICAicmVjaXBlIjogewogICAgIm1vZGVsIjogIlF3ZW4vUXdlbjIuNS1WTC0zQi1JbnN0cnVjdCIsCiAgICAibW9kZWxfcmV2aXNpb24iOiAiNjYyODU1NDZkMmI4MjFjZjQyMWQ0ZjVlYjI1NzYzNTlkMzc3MGNkMyIsCiAgICAic2Z0IjogeyJhZGFwdGVyIjogInRhc2stc3BlY2lmaWMgTG9SQSBhZGFwdGVyIiwgImJhdGNoX3NpemUiOiA0LCAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIjogOCwgImVwb2NocyI6IDUuMCwgInNjaGVkdWxlciI6ICJjb3NpbmUiLCAid2FybXVwX3N0ZXBzIjogMCwgIm1heF9ncmFkX25vcm0iOiAxLjAsICJwcmVjaXNpb24iOiAiYmYxNiIsICJsb3JhX3JhbmsiOiA4LCAibG9yYV9hbHBoYSI6IDMyLCAibG9yYV9kcm9wb3V0IjogMC4wNSwgInRhcmdldF9tb2R1bGVzIjogImFsbCJ9LAogICAgImRwbyI6IHsiZnJlc2hfYWRhcHRlciI6IHRydWUsICJyZWZlcmVuY2UiOiAiZnJvemVuIHBvc3QtU0ZUIGFkYXB0ZXIsIG5vIERQTyBhZGFwdGVyIiwgImJhdGNoX3NpemUiOiA0LCAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIjogOCwgImVwb2NocyI6IDUuMCwgInNjaGVkdWxlciI6ICJjb3NpbmUiLCAid2FybXVwX3N0ZXBzIjogMCwgIm1heF9ncmFkX25vcm0iOiAxLjAsICJiZXRhMCI6IDAuMSwgImxvc3MiOiAic2lnbW9pZCIsICJub2lzZV9zaWdtYSI6IDAuMDUsICJiZXRhX2Zsb29yIjogMS4wZS02LCAiZHJvcF9sYXN0IjogdHJ1ZX0KICB9LAogICJzZWVkcyI6IFsyMDI0LCAyMDI1LCAyMDI2XSwKICAid2VpZ2h0cyI6IFsxLjAsIDAuMCwgMC41XSwKICAiY29udHJvbHMiOiB7ImltYWdlX2NvbmRpdGlvbnMiOiBbInJlYWwiLCAic2h1ZmZsZSJdLCAic2h1ZmZsZSI6ICJmaXhlZC1wb2ludC1mcmVlIGZyZXF1ZW5jeS1zdHJhdGlmaWVkIGl0ZW0gYmlqZWN0aW9uIGJ1aWx0IGZyb20gdHJhaW5pbmcgcm93cyBvbmx5IiwgImNvbnN0YW50X2hhcmRuZXNzIjogInZhbGlkYXRpb24tb25seSBtZWFuLW1hdGNoZWQgZGlhZ25vc3RpYzsgbm90IGEgcHJpbWFyeSB0ZXN0IGFybSJ9LAogICJzbW9rZSI6IHsicHVycG9zZSI6ICJjb3JyZWN0bmVzcyBhbmQgbWVhc3VyZWQgcmVzb3VyY2Ugc21va2Ugb25seSIsICJ0cmFpbl9wYWlycyI6IDgsICJ2YWxpZGF0aW9uX3VzZXJzIjogNCwgInRlc3RfdXNlcnMiOiA0LCAiZXBvY2hzIjogMSwgIm1heF9vcHRpbWl6ZXJfdXBkYXRlcyI6IDIsICJtdXN0X25vdF9iZV91c2VkX2FzX3NpZ25hbCI6IHRydWV9LAogICJpbnZhcmlhbnRzIjogWyJ0cmFpbmluZyBwYWlyIGFuZCBoYXJkbmVzcyBoYXNoZXMgZG8gbm90IGNoYW5nZSB3aGVuIGhlbGQtb3V0IHRhcmdldHMvY2FuZGlkYXRlcy9pbWFnZXMgY2hhbmdlIiwgIm5lZ2F0aXZlIGlzIG5vdCB0YXJnZXQsIG9ic2VydmVkIGhpc3RvcnksIG9yIHRyYWluLWtub3duIGZ1dHVyZSBpdGVtIiwgInNodWZmbGUgaXMgYSBiaWplY3Rpb24gYW5kIGhhcyBubyBmaXhlZCBwb2ludHMiLCAiZ2xvYmFsIHJlc3BvbnNpdmVuZXNzIGJhdGNoIGhhcyBhdCBsZWFzdCB0aHJlZSBleGFtcGxlcyIsICJvbmx5IHRoZSBmcmVzaCBEUE8gYWRhcHRlciBjaGFuZ2VzIGR1cmluZyBEUE8iLCAiU0ZULW9ubHkgcmVmZXJlbmNlIGlzIGludmFyaWFudCB0byBEUE8gdXBkYXRlcyBhbmQgTm9ETyBub2lzZSIsICJwYXJ0aWFsIG9yIG1pc3Npbmcgc3RhZ2VzIGNhbm5vdCBiZSBsYWJlbGxlZCBDT01QTEVURSIsICJzb3VyY2UvY29uZmlnL2RhdGEvcGFyZW50IGhhc2hlcyBhcmUgcGVyc2lzdGVkIGFuZCBjaGVja2VkIl0sCiAgInN0YXRpc3RpY3MiOiB7ImJvb3RzdHJhcF9yZXNhbXBsZXMiOiAyMDAwMCwgImJvb3RzdHJhcF9zZWVkIjogMjAyNjA5MjAsICJob2xtX3ByaW1hcnlfY29udHJhc3RzIjogMiwgInRyYWluaW5nX3NlZWRfdW5jZXJ0YWludHkiOiAicmVwb3J0IGluZGl2aWR1YWwgc2VlZHMgYW5kIHNwcmVhZDsgYm9vdHN0cmFwIGlzIGNvbmRpdGlvbmFsIG9uIGZpdHRlZCBzZWVkIG1vZGVscyJ9Cn0K","test_hanorec_fidelity.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwppbXBvcnQgdW5pdHRlc3QKZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCmltcG9ydCB0cmFpbiBhcyB0cmFpbl9tb2R1bGUgICMgbm9xYTogRTQwMgpmcm9tIHRyYWluIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIF9hY2N1bXVsYXRpb25fZ3JvdXBfc2l6ZSwKICAgIF9jdWRhX3BlYWtfbWVtb3J5LAogICAgX2J1aWxkX2Rwb19leGFtcGxlcywKICAgIF9jaHVua2VkLAogICAgX2Zpbml0ZSwKICAgIF9qc29uX2RlZmF1bHQsCiAgICBfcmVzZXRfY3VkYV9wZWFrX21lbW9yeSwKICAgIF9zY29yZV9yb3dzLAogICAgX3Jlc3RvcmVfYWN0aXZlX2FkYXB0ZXJzLAogICAgX3NldF9hY3RpdmVfYWRhcHRlcnMsCiAgICBfc2V0X29ubHlfdHJhaW5hYmxlX2FkYXB0ZXIsCiAgICBoYXJzX25vcm1hbGl6ZV9oYXJkbmVzcywKICAgIHJlc3BvbnNpdmVuZXNzLAopCgpjbGFzcyBfQWRhcHRlckJhc2U6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWwpOgogICAgICAgIHNlbGYubW9kZWwgPSBtb2RlbAogICAgICAgIHNlbGYuY2FsbHMgPSBbXQoKICAgIGRlZiBzZXRfYWRhcHRlcihzZWxmLCBhZGFwdGVycyk6CiAgICAgICAgbmFtZXMgPSBbYWRhcHRlcnNdIGlmIGlzaW5zdGFuY2UoYWRhcHRlcnMsIHN0cikgZWxzZSBsaXN0KGFkYXB0ZXJzKQogICAgICAgIHNlbGYuY2FsbHMuYXBwZW5kKG5hbWVzKQogICAgICAgIGZvciBuYW1lLCBwYXJhbWV0ZXIgaW4gc2VsZi5tb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICAgICAgICAgIHBhcmFtZXRlci5yZXF1aXJlc19ncmFkID0gYW55KGYiLnthZGFwdGVyfS4iIGluIG5hbWUgZm9yIGFkYXB0ZXIgaW4gbmFtZXMpCgpjbGFzcyBfUGFyYW1ldGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYucmVxdWlyZXNfZ3JhZCA9IEZhbHNlCgoKY2xhc3MgX0FkYXB0ZXJNb2RlbDoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLmNhbGxzID0gW10KICAgICAgICBzZWxmLmFjdGl2ZV9hZGFwdGVyID0gTm9uZQogICAgICAgIHNlbGYucGVmdF9jb25maWcgPSB7InNmdCI6IG9iamVjdCgpLCAiZHBvIjogb2JqZWN0KCl9CiAgICAgICAgc2VsZi5wYXJhbWV0ZXJzID0gewogICAgICAgICAgICAibGF5ZXIuc2Z0LmxvcmFfQSI6IF9QYXJhbWV0ZXIoKSwKICAgICAgICAgICAgImxheWVyLmRwby5sb3JhX0EiOiBfUGFyYW1ldGVyKCksCiAgICAgICAgfQogICAgICAgIHNlbGYuYmFzZV9tb2RlbCA9IF9BZGFwdGVyQmFzZShzZWxmKQoKICAgIGRlZiBzZXRfYWRhcHRlcihzZWxmLCBhZGFwdGVyKToKICAgICAgICBzZWxmLmNhbGxzLmFwcGVuZChhZGFwdGVyKQoKICAgIGRlZiBuYW1lZF9wYXJhbWV0ZXJzKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLnBhcmFtZXRlcnMuaXRlbXMoKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGFjdGl2ZV9wZWZ0X2NvbmZpZyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5wZWZ0X2NvbmZpZ1tzZWxmLmFjdGl2ZV9hZGFwdGVyXQoKCmNsYXNzIF9DdWRhUHJvcGVydGllczoKICAgIHRvdGFsX21lbW9yeSA9IDFfMDAwCgoKY2xhc3MgX0Zha2VDdWRhOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYucmVzZXQgPSBGYWxzZQoKICAgIGRlZiBpc19hdmFpbGFibGUoc2VsZik6CiAgICAgICAgcmV0dXJuIFRydWUKCiAgICBkZWYgc3luY2hyb25pemUoc2VsZik6CiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgcmVzZXRfcGVha19tZW1vcnlfc3RhdHMoc2VsZik6CiAgICAgICAgc2VsZi5yZXNldCA9IFRydWUKCiAgICBkZWYgY3VycmVudF9kZXZpY2Uoc2VsZik6CiAgICAgICAgcmV0dXJuIDAKCiAgICBkZWYgZ2V0X2RldmljZV9wcm9wZXJ0aWVzKHNlbGYsIF9kZXZpY2UpOgogICAgICAgIHJldHVybiBfQ3VkYVByb3BlcnRpZXMoKQoKICAgIGRlZiBnZXRfZGV2aWNlX25hbWUoc2VsZiwgX2RldmljZSk6CiAgICAgICAgcmV0dXJuICJmYWtlLWdwdSIKCiAgICBkZWYgbWF4X21lbW9yeV9hbGxvY2F0ZWQoc2VsZiwgX2RldmljZSk6CiAgICAgICAgcmV0dXJuIDYwMAoKICAgIGRlZiBtYXhfbWVtb3J5X3Jlc2VydmVkKHNlbGYsIF9kZXZpY2UpOgogICAgICAgIHJldHVybiA3NTAKCgpjbGFzcyBfRmFrZVRvcmNoOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYuY3VkYSA9IF9GYWtlQ3VkYSgpCgoKCmNsYXNzIF9Ob0dyYWQ6CiAgICBkZWYgX19lbnRlcl9fKHNlbGYpOgogICAgICAgIHJldHVybiBOb25lCgogICAgZGVmIF9fZXhpdF9fKHNlbGYsICpfYXJncyk6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKY2xhc3MgX1Njb3JlVG9yY2g6CiAgICBkZWYgbm9fZ3JhZChzZWxmKToKICAgICAgICByZXR1cm4gX05vR3JhZCgpCgoKY2xhc3MgX1Njb3JlTW9kZWw6CiAgICBkZWYgX19pbml0X18oc2VsZik6CiAgICAgICAgc2VsZi50cmFpbmluZyA9IFRydWUKCiAgICBkZWYgZXZhbChzZWxmKToKICAgICAgICBzZWxmLnRyYWluaW5nID0gRmFsc2UKCiAgICBkZWYgdHJhaW4oc2VsZik6CiAgICAgICAgc2VsZi50cmFpbmluZyA9IFRydWUKCgpjbGFzcyBfU2NvcmVWYWx1ZToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB2YWx1ZSk6CiAgICAgICAgc2VsZi52YWx1ZSA9IGZsb2F0KHZhbHVlKQoKICAgIGRlZiBfX3N1Yl9fKHNlbGYsIG90aGVyKToKICAgICAgICByZXR1cm4gX1Njb3JlVmFsdWUoc2VsZi52YWx1ZSAtIG90aGVyLnZhbHVlKQoKICAgIGRlZiBjcHUoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYudmFsdWUKY2xhc3MgX1NjYWxhcjoKICAgIGRlZiBpdGVtKHNlbGYpOgogICAgICAgIHJldHVybiAxLjUKCgpjbGFzcyBIYU5vUmVjRmlkZWxpdHlUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF91cHN0cmVhbV9yZXNwb25zaXZlbmVzc19oYXNfZ2xvYmFsX3RyaW1tZWRfYmF0Y2hfc2VtYW50aWNzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdmFsdWUgPSByZXNwb25zaXZlbmVzcyhbMS4wLCAyLjAsIDMuMCwgMTAwLjBdKQogICAgICAgIHNlbGYuYXNzZXJ0R3JlYXRlcih2YWx1ZSwgMC4wKQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyh2YWx1ZSwgMi4wKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIHJlc3BvbnNpdmVuZXNzKFsxLjAsIGZsb2F0KCJuYW4iKSwgMy4wXSkKCiAgICBkZWYgdGVzdF9kcG9fZXhhbXBsZXNfcHJlc2VydmVfYmluYXJ5X3BhaXJfb3JkZXIoc2VsZikgLT4gTm9uZToKICAgICAgICBleGFtcGxlcyA9IF9idWlsZF9kcG9fZXhhbXBsZXMoCiAgICAgICAgICAgIFt7Imhpc3RvcnkiOiBbMSwgMiwgM10sICJwb3NpdGl2ZSI6IDQsICJuZWdhdGl2ZSI6IDV9XSwKICAgICAgICAgICAgeWVzX3Rva2VuX2lkPTEwLAogICAgICAgICAgICBub190b2tlbl9pZD0xMSwKICAgICAgICAgICAgc2h1ZmZsZV9tYXA9Tm9uZSwKICAgICAgICApCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgKICAgICAgICAgICAgWyhyb3dbImNhbmRpZGF0ZSJdLCByb3dbImNob3Nlbl90b2tlbiJdLCByb3dbInJlamVjdGVkX3Rva2VuIl0pIGZvciByb3cgaW4gZXhhbXBsZXNdLAogICAgICAgICAgICBbKDQsIDEwLCAxMSksICg1LCAxMSwgMTApXSwKICAgICAgICApCgogICAgZGVmIHRlc3RfYmF0Y2hfYm91bmRhcmllc19hbmRfaGFyZG5lc3NfYXJlX2ZhaWxfY2xvc2VkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChbbGVuKGJhdGNoKSBmb3IgYmF0Y2ggaW4gX2NodW5rZWQobGlzdChyYW5nZSgxMCkpLCA0LCBUcnVlKV0sIFs0LCA0XSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKFtsZW4oYmF0Y2gpIGZvciBiYXRjaCBpbiBfY2h1bmtlZChsaXN0KHJhbmdlKDEwKSksIDQsIEZhbHNlKV0sIFs0LCA0LCAyXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihoYXJzX25vcm1hbGl6ZV9oYXJkbmVzcyhbMC4wLCAxLjAsIDIuMF0pKSwgMykKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBfY2h1bmtlZChbMSwgMl0sIDAsIFRydWUpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgX2Zpbml0ZShbXSwgInZhbHVlcyIpCgogICAgZGVmIHRlc3RfbXVsdGlfYWRhcHRlcl9hY3RpdmF0aW9uX2ZyZWV6ZXNfc2Z0X3BhcmFtZXRlcnMoc2VsZikgLT4gTm9uZToKICAgICAgICBtb2RlbCA9IF9BZGFwdGVyTW9kZWwoKQogICAgICAgIF9zZXRfYWN0aXZlX2FkYXB0ZXJzKG1vZGVsLCBbInNmdCIsICJkcG8iXSkKICAgICAgICBfc2V0X29ubHlfdHJhaW5hYmxlX2FkYXB0ZXIobW9kZWwsICJkcG8iKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UobW9kZWwucGFyYW1ldGVyc1sibGF5ZXIuc2Z0LmxvcmFfQSJdLnJlcXVpcmVzX2dyYWQpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKG1vZGVsLnBhcmFtZXRlcnNbImxheWVyLmRwby5sb3JhX0EiXS5yZXF1aXJlc19ncmFkKQogICAgICAgIF9yZXN0b3JlX2FjdGl2ZV9hZGFwdGVycyhtb2RlbCwgWyJzZnQiLCAiZHBvIl0pCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShtb2RlbC5wYXJhbWV0ZXJzWyJsYXllci5zZnQubG9yYV9BIl0ucmVxdWlyZXNfZ3JhZCkKICAgICAgICBzZWxmLmFzc2VydFRydWUobW9kZWwucGFyYW1ldGVyc1sibGF5ZXIuZHBvLmxvcmFfQSJdLnJlcXVpcmVzX2dyYWQpCgogICAgZGVmIHRlc3RfbXVsdGlfYWRhcHRlcl9hY3RpdmF0aW9uX3VzZXNfcGVmdF9iYXNlX21vZGVsKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBfQWRhcHRlck1vZGVsKCkKICAgICAgICBfc2V0X2FjdGl2ZV9hZGFwdGVycyhtb2RlbCwgWyJzZnQiLCAiZHBvIl0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChtb2RlbC5iYXNlX21vZGVsLmNhbGxzLCBbWyJzZnQiLCAiZHBvIl1dKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobW9kZWwuYWN0aXZlX2FkYXB0ZXIsICJkcG8iKQogICAgICAgIHNlbGYuYXNzZXJ0SXMobW9kZWwuYWN0aXZlX3BlZnRfY29uZmlnLCBtb2RlbC5wZWZ0X2NvbmZpZ1siZHBvIl0pCiAgICAgICAgX3NldF9hY3RpdmVfYWRhcHRlcnMobW9kZWwsICJzZnQiKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobW9kZWwuY2FsbHMsIFsic2Z0Il0pCiAgICBkZWYgdGVzdF9wYXJ0aWFsX2FjY3VtdWxhdGlvbl91c2VzX29uZV9ncm91cF9kaXZpc29yKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgKICAgICAgICAgICAgW19hY2N1bXVsYXRpb25fZ3JvdXBfc2l6ZShwb3NpdGlvbiwgMTAsIDQpIGZvciBwb3NpdGlvbiBpbiByYW5nZSgxMCldLAogICAgICAgICAgICBbNCwgNCwgNCwgNCwgNCwgNCwgNCwgNCwgMiwgMl0sCiAgICAgICAgKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIF9hY2N1bXVsYXRpb25fZ3JvdXBfc2l6ZSgwLCAwLCA0KQoKCiAgICBkZWYgdGVzdF9jdWRhX3BlYWtfbWVtb3J5X3JlcG9ydHNfY2FwYWNpdHlfZnJhY3Rpb25zKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdG9yY2hfbW9kdWxlID0gX0Zha2VUb3JjaCgpCiAgICAgICAgX3Jlc2V0X2N1ZGFfcGVha19tZW1vcnkodG9yY2hfbW9kdWxlKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZSh0b3JjaF9tb2R1bGUuY3VkYS5yZXNldCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKAogICAgICAgICAgICBfY3VkYV9wZWFrX21lbW9yeSh0b3JjaF9tb2R1bGUpLAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiZGV2aWNlIjogImZha2UtZ3B1IiwKICAgICAgICAgICAgICAgICJ0b3RhbF9ieXRlcyI6IDFfMDAwLAogICAgICAgICAgICAgICAgInBlYWtfYWxsb2NhdGVkX2J5dGVzIjogNjAwLAogICAgICAgICAgICAgICAgInBlYWtfcmVzZXJ2ZWRfYnl0ZXMiOiA3NTAsCiAgICAgICAgICAgICAgICAicGVha19hbGxvY2F0ZWRfZnJhY3Rpb24iOiAwLjYsCiAgICAgICAgICAgICAgICAicGVha19yZXNlcnZlZF9mcmFjdGlvbiI6IDAuNzUsCiAgICAgICAgICAgIH0sCiAgICAgICAgKQoKICAgIGRlZiB0ZXN0X2NhbmRpZGF0ZV9zY29yaW5nX2JhdGNoZXNfd2l0aG91dF9jaGFuZ2luZ19yYW5rX2NvbnRyYWN0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBfU2NvcmVNb2RlbCgpCiAgICAgICAgYmF0Y2hfc2l6ZXMgPSBbXQoKICAgICAgICBkZWYgZmFrZV9pbnB1dHMoX3Byb2Nlc3NvciwgX21vZGVsLCBfY2F0YWxvZywgZXhhbXBsZXMsIF9waXhlbHMsIF9zaHVmZmxlKToKICAgICAgICAgICAgYmF0Y2hfc2l6ZXMuYXBwZW5kKGxlbihleGFtcGxlcykpCiAgICAgICAgICAgIHJldHVybiBleGFtcGxlcwoKICAgICAgICBkZWYgZmFrZV9sb2dwcm9icyhfbW9kZWwsIGV4YW1wbGVzLCBfZnVuY3Rpb25hbCwgX3llcywgX25vKToKICAgICAgICAgICAgcmV0dXJuIFsoX1Njb3JlVmFsdWUoZXhhbXBsZVsiY2FuZGlkYXRlIl0pLCBfU2NvcmVWYWx1ZSgwLjApKSBmb3IgZXhhbXBsZSBpbiBleGFtcGxlc10KCiAgICAgICAgd2l0aCAoCiAgICAgICAgICAgIHBhdGNoLm9iamVjdCh0cmFpbl9tb2R1bGUsICJfYnVpbGRfYmF0Y2hfaW5wdXRzIiwgc2lkZV9lZmZlY3Q9ZmFrZV9pbnB1dHMpLAogICAgICAgICAgICBwYXRjaC5vYmplY3QodHJhaW5fbW9kdWxlLCAiX2JhdGNoX2Fuc3dlcl9sb2dwcm9icyIsIHNpZGVfZWZmZWN0PWZha2VfbG9ncHJvYnMpLAogICAgICAgICk6CiAgICAgICAgICAgIHJlc3VsdCA9IF9zY29yZV9yb3dzKAogICAgICAgICAgICAgICAgW3sKICAgICAgICAgICAgICAgICAgICAidXNlcl9pZCI6ICJ1IiwKICAgICAgICAgICAgICAgICAgICAiaGlzdG9yeSI6IFs5LCA4LCA3XSwKICAgICAgICAgICAgICAgICAgICAidGFyZ2V0IjogMiwKICAgICAgICAgICAgICAgICAgICAiY2FuZGlkYXRlcyI6IFsxLCAzLCAyXSwKICAgICAgICAgICAgICAgIH1dLAogICAgICAgICAgICAgICAgb2JqZWN0KCksIG1vZGVsLCB7fSwgODE5MiwgMSwgMCwgX1Njb3JlVG9yY2goKSwgb2JqZWN0KCksIE5vbmUsCiAgICAgICAgICAgICAgICBldmFsX2JhdGNoX3NpemU9MiwKICAgICAgICAgICAgKQoKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGJhdGNoX3NpemVzLCBbMiwgMV0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbMF1bInJhbmtlZF9jYW5kaWRhdGVzIl0sIFszLCAyLCAxXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdFswXVsicmFuayJdLCAyKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShtb2RlbC50cmFpbmluZykKCiAgICBkZWYgdGVzdF9qc29uX2NvbnZlcnRlcl9oYW5kbGVzX3NjYWxhcl9wcm90b2NvbF9hbmRfcmVqZWN0c191bmtub3duX3ZhbHVlcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoX2pzb25fZGVmYXVsdChfU2NhbGFyKCkpLCAxLjUpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhUeXBlRXJyb3IpOgogICAgICAgICAgICBfanNvbl9kZWZhdWx0KG9iamVjdCgpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK","test_hanorec_protocol.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHVuaXR0ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKZnJvbSBhdWRpdCBpbXBvcnQgYXVkaXRfYXJ0aWZhY3RzLCBwcm90b2NvbF9lcnJvcnMgICMgbm9xYTogRTQwMgpmcm9tIHByZXAgaW1wb3J0IF9idWlsZF9mcmVxdWVuY3lfbWF0Y2hlZF9kZXJhbmdlbWVudCwgX3NlZWRlZF9zZWxlY3Rpb24gICMgbm9xYTogRTQwMgoKUFJPVE9DT0wgPSBST09UIC8gInJ1bi1wcm90b2NvbC5qc29uIgoKCmNsYXNzIEhhTm9SZWNQcm90b2NvbFRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3Byb3RvY29sX2lzX2NvbXBsZXRlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcHJvdG9jb2wgPSBqc29uLmxvYWRzKFBST1RPQ09MLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb3RvY29sX2Vycm9ycyhwcm90b2NvbCksIFtdKQoKICAgIGRlZiB0ZXN0X3NlbGVjdGlvbl9pc19zZWVkZWRfYW5kX25vdF9wcmVmaXhfYmlhc2VkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm93cyA9IFsoaW5kZXgsIFtpbmRleCwgaW5kZXggKyAxLCBpbmRleCArIDIsIGluZGV4ICsgM10pIGZvciBpbmRleCBpbiByYW5nZSgyMCldCiAgICAgICAgZmlyc3QgPSBfc2VlZGVkX3NlbGVjdGlvbihyb3dzLCA1LCAyMDI0LCAidHJhaW4iKQogICAgICAgIHNlY29uZCA9IF9zZWVkZWRfc2VsZWN0aW9uKHJvd3MsIDUsIDIwMjQsICJ0cmFpbiIpCiAgICAgICAgb3RoZXIgPSBfc2VlZGVkX3NlbGVjdGlvbihyb3dzLCA1LCAyMDI1LCAidHJhaW4iKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmlyc3QsIHNlY29uZCkKICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKGZpcnN0LCBvdGhlcikKICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKFtyb3dbMF0gZm9yIHJvdyBpbiBmaXJzdF0sIGxpc3QocmFuZ2UoNSkpKQoKICAgIGRlZiB0ZXN0X2ZyZXF1ZW5jeV9zaHVmZmxlX2lzX2JpamVjdGl2ZV9hbmRfZml4ZWRfcG9pbnRfZnJlZShzZWxmKSAtPiBOb25lOgogICAgICAgIGl0ZW1faWRzID0gbGlzdChyYW5nZSgxLCA5KSkKICAgICAgICBmcmVxdWVuY2llcyA9IHsxOiA1LCAyOiA1LCAzOiA0LCA0OiA0LCA1OiAzLCA2OiAzLCA3OiAyLCA4OiAyfQogICAgICAgIG1hcHBpbmcgPSBfYnVpbGRfZnJlcXVlbmN5X21hdGNoZWRfZGVyYW5nZW1lbnQoaXRlbV9pZHMsIGZyZXF1ZW5jaWVzLCAyMDI0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KG1hcHBpbmcpLCBzZXQoaXRlbV9pZHMpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KG1hcHBpbmcudmFsdWVzKCkpLCBzZXQoaXRlbV9pZHMpKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoc291cmNlICE9IHRhcmdldCBmb3Igc291cmNlLCB0YXJnZXQgaW4gbWFwcGluZy5pdGVtcygpKSkKCiAgICBkZWYgdGVzdF9hcnRpZmFjdF9hdWRpdF9yZWplY3RzX3plcm9fYXJtcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHByb3RvY29sID0ganNvbi5sb2FkcyhQUk9UT0NPTC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0ZW1wb3Jhcnk6CiAgICAgICAgICAgIHJvb3QgPSBQYXRoKHRlbXBvcmFyeSkKICAgICAgICAgICAgKHJvb3QgLyAic2Z0X2J1bmRsZS5qc29uIikud3JpdGVfdGV4dCgie30iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICAocm9vdCAvICJzZnRfbG9yYV9zdGF0ZS5wdCIpLndyaXRlX2J5dGVzKGIic3RhdGUiKQogICAgICAgICAgICByZXN1bHQgPSBhdWRpdF9hcnRpZmFjdHMocm9vdCwgcHJvdG9jb2wpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJzdGF0dXMiXSwgIkZBSUwiKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoYW55KCJhcm0gbWF0cml4IG1pc21hdGNoIiBpbiBlcnJvciBmb3IgZXJyb3IgaW4gcmVzdWx0WyJlcnJvcnMiXSkpCgogICAgZGVmIHRlc3Rfc21va2VfYXVkaXRfcmVqZWN0c19taXNzaW5nX2NvbXB1dGVfdGVsZW1ldHJ5KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcHJvdG9jb2wgPSBqc29uLmxvYWRzKFBST1RPQ09MLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRlbXBvcmFyeToKICAgICAgICAgICAgcm9vdCA9IFBhdGgodGVtcG9yYXJ5KQogICAgICAgICAgICAocm9vdCAvICJzZnRfYnVuZGxlLmpzb24iKS53cml0ZV90ZXh0KCJ7fSIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIChyb290IC8gInNmdF9sb3JhX3N0YXRlLnB0Iikud3JpdGVfYnl0ZXMoYiJzdGF0ZSIpCiAgICAgICAgICAgIChyb290IC8gImFybS5wdCIpLndyaXRlX2J5dGVzKGIiY2hlY2twb2ludCIpCiAgICAgICAgICAgIChyb290IC8gImFybV9yZXN1bHQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHsKICAgICAgICAgICAgICAgICAgICAid2VpZ2h0IjogMS4wLAogICAgICAgICAgICAgICAgICAgICJpbWFnZV9jb25kaXRpb24iOiAicmVhbCIsCiAgICAgICAgICAgICAgICAgICAgInN0YXR1cyI6ICJDT01QTEVURSIsCiAgICAgICAgICAgICAgICAgICAgInByZWRpY3Rpb25zIjogW3sKICAgICAgICAgICAgICAgICAgICAgICAgIm5kY2dAMTAiOiAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICJyZWNhbGxAMTAiOiAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICJjYW5kaWRhdGVfcmVjYWxsQDIwIjogMC4wLAogICAgICAgICAgICAgICAgICAgIH1dLAogICAgICAgICAgICAgICAgICAgICJjaGVja3BvaW50IjogImFybS5wdCIsCiAgICAgICAgICAgICAgICAgICAgInBhcmVudF9zZnRfc2hhMjU2IjogImhhc2giLAogICAgICAgICAgICAgICAgfSksCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIChyb290IC8gInNtb2tlX21hbmlmZXN0Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcyh7CiAgICAgICAgICAgICAgICAgICAgInB1cnBvc2UiOiAiY29ycmVjdG5lc3Nfb25seV9ub25fc2lnbmFsIiwKICAgICAgICAgICAgICAgICAgICAibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiOiBUcnVlLAogICAgICAgICAgICAgICAgfSksCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlc3VsdCA9IGF1ZGl0X2FydGlmYWN0cyhyb290LCBwcm90b2NvbCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbInN0YXR1cyJdLCAiRkFJTCIpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oImludmFsaWQgc2Z0X2dwdV9tZW1vcnkgdGVsZW1ldHJ5IiwgcmVzdWx0WyJlcnJvcnMiXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRJbigiaW52YWxpZCBkcG9fc3RhZ2VfdGltaW5ncyBvcHRpbWl6ZXIgdGltaW5nIiwgcmVzdWx0WyJlcnJvcnMiXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg=="}
EXPECTED_SOURCE_SHA256 = {"audit.py":"e48b89cbbd50d760724b308c3bc1773bf022740e67bc3c41d5436ab2954393c3","experiment.json":"498662ade673a296e6501e44b5585f9914b58613d334a3a92f32af30ab9f3f09","prep.py":"6a82314f6b90dcc50d4d2dca7f7e8d195f60c57d4752ccda28df429dbef379ff","run-protocol.json":"237d946eea9434f647c45449393b37839f3d07f9fb524d699681fb973d3411e1","test_hanorec_fidelity.py":"e56f3d71902d87a5c803ec4b0266b01323a943967b69c0e3d83f1bd335d3dd0e","test_hanorec_protocol.py":"f7ba56e9091420c26dea5181672d2f3e470ddd771d8c88f35daf4617ae76cd29","train.py":"808305424198a841928c2ab6f4906df5c3aec31ebd5606f5fc05832d9c621c0d"}
for name, encoded in EMBEDDED_SOURCES.items():
    payload = base64.b64decode(encoded)
    actual = hashlib.sha256(payload).hexdigest()
    assert actual == EXPECTED_SOURCE_SHA256[name], f"embedded source hash mismatch: {name}"
    (SOURCE_DIR / name).write_bytes(payload)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
print("Verified embedded sources:", json.dumps(EXPECTED_SOURCE_SHA256, indent=2))


## 2. Kiểm tra CUDA / BF16 trước khi tải model

Cell tương thích với Colab torch CUDA 12/13, gồm nạp NVRTC builtins trước phép thử GPU. T4 không đáp ứng protocol BF16 native của code hiện tại.


In [ ]:
import ctypes, glob, os, subprocess, sys
import torch

assert torch.cuda.is_available(), "Chọn Runtime GPU trước khi tiếp tục."
cuda_major = (torch.version.cuda or "0").split(".")[0]
site_dirs = sorted({p for p in sys.path if p.endswith(("site-packages", "dist-packages"))})
def find_nvrtc_builtins():
    roots = [os.path.join(os.path.dirname(torch.__file__), "lib")] + [os.path.join(s, "nvidia") for s in site_dirs]
    hits = []
    for root in roots:
        hits += glob.glob(os.path.join(root, "**", f"libnvrtc-builtins.so.{cuda_major}*"), recursive=True)
    return sorted(set(hits))
hits = find_nvrtc_builtins()
if not hits:
    package = "nvidia-cuda-nvrtc" if cuda_major == "13" else f"nvidia-cuda-nvrtc-cu{cuda_major}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    hits = find_nvrtc_builtins()
assert hits, f"Không tìm thấy NVRTC builtins cho CUDA {cuda_major}; không chạy GPU test."
for path in hits:
    ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)

gpu = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
bf16_ok = torch.cuda.is_bf16_supported()
print({"gpu": gpu, "vram_gb": round(props.total_memory / 1e9, 2), "compute_capability": cc,
       "torch": torch.__version__, "cuda": torch.version.cuda, "native_bf16": bf16_ok})
if not bf16_ok:
    raise RuntimeError("v11 protocol requires native BF16 (L4/A100-class). T4/FP16 fallback is intentionally disabled.")
x = torch.arange(1, 9, device="cuda", dtype=torch.bfloat16)
assert float(x.float().prod()) == 40320.0
assert torch.isfinite(x.float().exp2()).all()
print("CUDA/BF16 self-test passed")


## 3. Nạp input local và xác minh artifact

Notebook không mount Drive. Upload một ZIP chứa 9 artifact hoặc chọn các file rời khi cell yêu cầu. ZIP được kiểm tra đường dẫn trước khi giải nén. Có thể đặt sẵn file trong `/content/v11_inputs/`; notebook tìm trong thư mục con và fail nếu thiếu, trùng tên hoặc hash sai.


In [ ]:
from pathlib import Path
import hashlib, json, io, zipfile

INPUT_ROOT = Path("/content/v11_inputs")
RUNS_ROOT = Path("/content/v11_colab_runs")
IMAGE_CACHE_ROOT = Path("/content/v11_image_cache")
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

with (SOURCE_DIR / "experiment.json").open(encoding="utf-8") as f:
    BASE_CONFIG = json.load(f)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def missing_pins():
    return [name for name in BASE_CONFIG["artifact_pins"] if not any(p.is_file() for p in INPUT_ROOT.rglob(name))]

missing_artifacts = missing_pins()
if missing_artifacts:
    from google.colab import files
    print(f"Thiếu {len(missing_artifacts)}/9 artifact. Upload một ZIP chứa chúng hoặc chọn các file rời.")
    uploaded = files.upload()
    for filename, payload in uploaded.items():
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(payload)) as archive_file:
                root = INPUT_ROOT.resolve()
                for member in archive_file.infolist():
                    destination = (INPUT_ROOT / member.filename).resolve()
                    if destination != root and root not in destination.parents:
                        raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
                archive_file.extractall(INPUT_ROOT)
        else:
            (INPUT_ROOT / Path(filename).name).write_bytes(payload)
    del uploaded
    missing_artifacts = missing_pins()
if missing_artifacts:
    raise FileNotFoundError(
        f"Thiếu {len(missing_artifacts)}/9 artifact đã pin dưới {INPUT_ROOT}:\n"
        + "\n".join(f" - {name}" for name in missing_artifacts)
    )

resolved_inputs = {}
for filename, expected in BASE_CONFIG["artifact_pins"].items():
    matches = [p for p in INPUT_ROOT.rglob(filename) if p.is_file()]
    if len(matches) != 1:
        raise RuntimeError(f"{filename}: cần đúng 1 file dưới {INPUT_ROOT}, tìm thấy {len(matches)}")
    actual = sha256_file(matches[0])
    if actual != expected:
        raise RuntimeError(f"SHA-256 mismatch cho {filename}: expected={expected}, actual={actual}")
    resolved_inputs[filename] = str(matches[0])
    print("PIN OK", filename, f"({matches[0].stat().st_size / 1e6:.1f} MB)")
print("Toàn bộ artifact pins hợp lệ.")


## 4. Chạy CPU protocol/fidelity tests

Đây là regression tests của repo, không thay thế kiểm tra GPU thật bên dưới.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(SOURCE_DIR), "-p", "test_hanorec_*.py", "-v"],
    cwd=SOURCE_DIR, check=True
)


## 5. Cấu hình pilot và dựng dữ liệu

Pilot dùng nguyên data pins / split và model revision đã đóng băng. Thay đổi duy nhất để chạy nhanh hơn là một epoch, `max_pixels=4096`, hai DPO weights, và evaluation cohort thu nhỏ theo quy tắc đã định trước: chỉ chạy Qwen cho user có target trong top-20; mọi user vẫn có mặt trong mẫu số metric toàn validation.

`test_data.txt` được source prep đọc để dựng input object nhưng test rows bị loại trước mọi lần gọi SFT/DPO model.


In [ ]:
import copy, json, time
from pathlib import Path
from prep import prepare

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
OUT_ROOT = RUNS_ROOT / f"v11-pilot-{RUN_ID}"
OUT_ROOT.mkdir(parents=True, exist_ok=False)
image_link = OUT_ROOT / "images"
image_link.symlink_to(IMAGE_CACHE_ROOT, target_is_directory=True)
CONFIG = copy.deepcopy(BASE_CONFIG)
CONFIG.update({
    "seed": 2024,
    "train_pairs": 530,
    "validation_users": 265,
    "eval_users": 1,
    "history_items": 3,
    "smoke": False,
    "sft_epochs": 1,
    "dpo_epochs": 1,
    "sft_batch_size": 1,
    "batch_size": 3,
    "gradient_accumulation_steps": 2,
    "eval_batch_size": 2,
    "max_pixels": 4096,
    "weights": [1.0, 0.5],
    "budget_seconds": 21600,
    "verify_scoring_batching": True,
    "sft_image_condition": "real",
})
print("Pilot config:", json.dumps({k: CONFIG[k] for k in ("seed", "train_pairs", "validation_users", "sft_epochs", "dpo_epochs", "weights", "max_pixels", "budget_seconds")}, indent=2))

t0 = time.time()
prepared = prepare(CONFIG, INPUT_ROOT, OUT_ROOT, SOURCE_DIR)
ALL_VALIDATION_ROWS = list(prepared["validation"])
SASREC_VALIDATION = list(prepared["baselines"]["sasrec_validation"])
INFORMATIVE_ROWS = [row for row in ALL_VALIDATION_ROWS if int(row["target"]) in set(map(int, row["candidates"]))]
if not INFORMATIVE_ROWS:
    raise RuntimeError("Validation cohort has zero targets in frozen SASRec top-20; no reranker experiment is identifiable.")
# Save all-user denominator and SASRec baseline; only score informative rows with Qwen.
prepared["validation"] = INFORMATIVE_ROWS
prepared["evaluation"] = []
prepared["baselines"]["sasrec_test"] = []
print({"prepare_seconds": round(time.time()-t0, 1), "validation_users": len(ALL_VALIDATION_ROWS),
       "informative_users": len(INFORMATIVE_ROWS), "candidate_recall@20": len(INFORMATIVE_ROWS)/len(ALL_VALIDATION_ROWS),
       "train_pairs": len(prepared["train"]), "test_model_scoring": "disabled"})


## 6. Train one real-image SFT parent

SFT scores only the informative validation rows. `test` evaluation list is empty. Check output status and reference reproducibility before proceeding.

Trước SFT, cell áp dụng runtime patch đã dùng cho exploratory v11: ảnh sau resize có cạnh <28 được giữ tỉ lệ và letterbox trên canvas trắng 56×56; các ảnh khác không đổi. Điều chỉnh được ghi trong report.


In [ ]:
import train as train_module
from functools import lru_cache
from PIL import Image, ImageOps

PATCH_VERSION = "v11-qwen25vl-min28-letterbox56"
existing_resize = train_module._resize_image
base_resize = getattr(existing_resize, "_v11_base_resize", existing_resize)
if getattr(existing_resize, "_v11_patch_version", None) == PATCH_VERSION:
    patched_resize = existing_resize
    PATCH_ADJUSTMENTS = patched_resize._v11_adjustments
else:
    PATCH_ADJUSTMENTS = []
    def _min28_letterbox56(image):
        if min(image.size) >= 28:
            return image
        contained = ImageOps.contain(image, (56, 56), method=Image.Resampling.LANCZOS)
        canvas = Image.new("RGB", (56, 56), (255, 255, 255))
        canvas.paste(contained, ((56 - contained.width) // 2, (56 - contained.height) // 2))
        return canvas

    _probe = _min28_letterbox56(Image.new("RGB", (21, 186), (0, 0, 0)))
    assert _probe.size == (56, 56) and _probe.width * _probe.height <= CONFIG["max_pixels"]
    assert _min28_letterbox56(Image.new("RGB", (88, 46))).size == (88, 46)

    @lru_cache(maxsize=512)
    def patched_resize(image_path, max_pixels):
        image = base_resize(image_path, max_pixels)
        after_area_resize = image.size
        fixed = _min28_letterbox56(image)
        if fixed.size != after_area_resize:
            PATCH_ADJUSTMENTS.append({
                "image_path": str(image_path),
                "after_area_resize_size": list(after_area_resize),
                "final_size": list(fixed.size),
                "letterboxed": True,
            })
        return fixed

    patched_resize._v11_base_resize = base_resize
    patched_resize._v11_patch_version = PATCH_VERSION
    patched_resize._v11_adjustments = PATCH_ADJUSTMENTS
    train_module._resize_image = patched_resize
print({"image_preprocessing_patch": PATCH_VERSION, "cached_letterbox_adjustments": len(PATCH_ADJUSTMENTS)})
from train import run_sft
SFT_DIR = OUT_ROOT / "sft_real"
sft_result = run_sft(CONFIG, prepared, SFT_DIR, SOURCE_DIR)
assert sft_result["status"] == "COMPLETE"
assert len(sft_result["sft_validation_predictions"]) == len(INFORMATIVE_ROWS)
assert sft_result["sft_test_predictions"] == [], "Test scoring must remain disabled."
print({"sft_status": sft_result["status"], "sft_updates": sft_result["sft_optimizer_updates"],
       "reference_probe": sft_result["reference_reproducibility_check"], "output": str(SFT_DIR)})


## 7. Train two DPO arms

Both arms resume from the same hashed SFT parent. No shuffle arm in this quick pilot; therefore it cannot establish image-specific causal contribution.


In [ ]:
from train import run_arm
ARM_RESULTS = {}
for weight in (1.0, 0.5):
    arm_dir = OUT_ROOT / f"arm_w{weight:.1f}_real"
    result = run_arm(CONFIG, SFT_DIR, weight, "real", arm_dir, SOURCE_DIR)
    assert result["status"] == "COMPLETE"
    assert len(result["validation_predictions"]) == len(INFORMATIVE_ROWS)
    assert result["predictions"] == [], "Test scoring must remain disabled."
    ARM_RESULTS[weight] = result
    print({"weight": weight, "status": result["status"], "optimizer_updates": result["optimizer_updates"],
           "checkpoint_reloaded": result["checkpoint_reloaded"], "output": str(arm_dir)})


## 8. Validation-only summary and paired uncertainty

Metrics are expanded back to the complete validation cohort: users whose target is absent from top-20 contribute zero to NDCG/Recall. Bootstrap intervals are exploratory and conditional on this one training seed. They do not test seed-to-seed uncertainty.


In [ ]:
import math, random
import numpy as np

def full_cohort_metrics(predictions):
    pred_by_key = {(str(r["user_id"]), int(r["target"])): r for r in predictions}
    values = []
    for row in ALL_VALIDATION_ROWS:
        key = (str(row["user_id"]), int(row["target"]))
        values.append(pred_by_key.get(key, {"ndcg@10": 0.0, "recall@10": 0.0, "candidate_recall@20": 0.0}))
    return values

def summarize(values):
    return {metric: float(np.mean([float(v[metric]) for v in values])) for metric in ("ndcg@10", "recall@10", "candidate_recall@20")}

BASELINE_VALUES = SASREC_VALIDATION
SFT_VALUES = full_cohort_metrics(sft_result["sft_validation_predictions"])
W1_VALUES = full_cohort_metrics(ARM_RESULTS[1.0]["validation_predictions"])
W05_VALUES = full_cohort_metrics(ARM_RESULTS[0.5]["validation_predictions"])
rows = {"SASRec order": summarize(BASELINE_VALUES), "SFT-only": summarize(SFT_VALUES),
        "DPO w=1.0": summarize(W1_VALUES), "DPO w=0.5": summarize(W05_VALUES)}
print(json.dumps(rows, indent=2))

def paired_bootstrap(a, b, metric, n=20000, seed=20260924):
    deltas = [float(x[metric]) - float(y[metric]) for x, y in zip(a, b, strict=True)]
    rng = random.Random(seed)
    means = sorted(sum(deltas[rng.randrange(len(deltas))] for _ in deltas) / len(deltas) for _ in range(n))
    return {"mean_delta": float(np.mean(deltas)), "ci95": [means[int(0.025*n)-1], means[min(n-1, int(0.975*n))]],
            "positive_user_fraction": sum(x > 0 for x in deltas) / len(deltas), "users": len(deltas)}

contrasts = {
    "SFT-only minus SASRec": paired_bootstrap(SFT_VALUES, BASELINE_VALUES, "ndcg@10"),
    "DPO w=1.0 minus SASRec": paired_bootstrap(W1_VALUES, BASELINE_VALUES, "ndcg@10"),
    "DPO w=0.5 minus SASRec": paired_bootstrap(W05_VALUES, BASELINE_VALUES, "ndcg@10"),
    "DPO w=0.5 minus w=1.0": paired_bootstrap(W05_VALUES, W1_VALUES, "ndcg@10"),
}
print("Paired validation NDCG@10 contrasts:")
print(json.dumps(contrasts, indent=2))


## 9. Tải kết quả về máy

Metrics, cache và checkpoint lưu trong `/content`. Checkpoint SFT được dùng tiếp cho hai DPO arm trong cùng runtime. Tải archive cuối về máy để giữ kết quả sau khi Colab reset.


In [ ]:
import zipfile
from datetime import datetime, timezone

report = {
    "purpose": "single_seed_validation_pilot_non_confirmatory",
    "must_not_be_used_as_confirmation": True,
    "seed": CONFIG["seed"], "train_pairs": len(prepared["train"]),
    "validation_users": len(ALL_VALIDATION_ROWS), "informative_validation_users": len(INFORMATIVE_ROWS),
    "candidate_recall_at_20": len(INFORMATIVE_ROWS) / len(ALL_VALIDATION_ROWS),
    "test_model_scoring": "disabled", "config": {k: CONFIG[k] for k in ("sft_epochs", "dpo_epochs", "weights", "max_pixels", "batch_size", "gradient_accumulation_steps")},
    "runtime_preprocessing_patch": PATCH_VERSION,
    "letterboxed_image_count": len(PATCH_ADJUSTMENTS),
    "letterboxed_adjustments": PATCH_ADJUSTMENTS,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
report_path = OUT_ROOT / "validation_pilot_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
archive = Path("/content") / f"v11_colab_pilot_{RUN_ID}.zip"
include_names = ["sft_real", "arm_w1.0_real", "arm_w0.5_real", "validation_pilot_report.json"]
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in include_names:
        path = OUT_ROOT / name
        if path.is_dir():
            for file in path.rglob("*"):
                if file.is_file():
                    zf.write(file, file.relative_to(OUT_ROOT))
        elif path.is_file():
            zf.write(path, path.relative_to(OUT_ROOT))
print("Saved report:", report_path)
print("Saved archive:", archive)
from google.colab import files
files.download(str(archive))
